# WP3 Africa Hazard Ranking — Notebook 06
## EM-DAT + DesInventar joint exploration & extraction (2000–2024)

### What this notebook does
This notebook extracts **hazard-specific, country-level** indicators from **EM-DAT** and **DesInventar** for 2000–2024 and appends the *selected* indicators into:

`data/intermediate/wp3_country_indicators_long.csv`

Because EM-DAT and DesInventar may describe overlapping real-world disasters, we **avoid double counting** by selecting **one dataset per country × hazard × indicator**, automatically, based on a transparent **completeness_score**.

### Implemented assumptions (locked / agreed)
- Scope = **DataBank countries only**, using `data/raw/scope/ISO3_Regions.csv`
- Time window = **2000–2024 inclusive**
- We do **hazard-specific indicators only** 
- Per-capita denominators use **World Bank WDI SP.POP.TOTL**, reference **Pop2024**
- DesInventar economic losses:
  - We only append **USD** losses if we can clearly identify the **US$** field (DesInventar distinguishes local currency loss vs US$ equivalent)
  - Local-currency losses stay in QA exports but are **not appended** to master.

### EM-DAT notes relevant to extraction
- EM-DAT “Total Damage, Adjusted (‘000 US$)” is CPI-adjusted; “Admin Units” is a JSON array using FAO GAUL codes (Admin-1/2).
See EM-DAT documentation for economic adjustment and admin units definitions.

### Outputs (all written by the notebook)
Diagnostics:
- `data/intermediate/wp3_emdat_desinventar_coverage.csv`
- `data/intermediate/wp3_emdat_desinventar_indicator_completeness.csv`
- `data/intermediate/wp3_emdat_desinventar_selection_decisions.csv`
- `data/intermediate/wp3_emdat_desinventar_overlap_diagnostics.csv` (approximate)

Clean tables:
- `data/intermediate/emdat_clean_mapped_2000_2024.csv`
- `data/intermediate/desinventar_clean_mapped_2000_2024.csv`

Aggregates:
- `data/intermediate/wp3_emdat_country_hazard_indicators_2000_2024.csv`
- `data/intermediate/wp3_desinventar_country_hazard_indicators_2000_2024.csv`
- `data/intermediate/wp3_selected_country_hazard_indicators_2000_2024.csv`


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import math
import re
import zipfile
from hashlib import md5

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 200)


# 0) Paths and global settings

In [2]:
PROJECT_ROOT = Path(".").resolve()

# Standard repo paths
SCOPE_PATH = PROJECT_ROOT.parent / "data" / "raw" / "scope" / "ISO3_Regions.csv"
EMDAT_DIR  = PROJECT_ROOT.parent / "data" / "raw" / "EMDAT"
DESINV_DIR = PROJECT_ROOT.parent / "data" / "raw" / "desinventar_sendai"
POP_DIR    = PROJECT_ROOT.parent / "data" / "raw" / "population"

OUT_DIR = PROJECT_ROOT.parent / "data" / "intermediate"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_PATH = PROJECT_ROOT.parent / "data" / "intermediate" / "wp3_country_indicators_long.csv"
MASTER_PATH.parent.mkdir(parents=True, exist_ok=True)

# fallbacks
FALLBACKS = {
    "SCOPE": Path("/mnt/data/ISO3_Regions.csv"),
    "EMDAT": Path("/mnt/data/emdat_public.xlsx"),
    "DESINV": Path("/mnt/data/desinventar_combined_translated_grouped.csv"),
    "WDI_ZIP": Path("/mnt/data/API_SP.POP.TOTL_DS2_en_csv_v2_34.zip"),
}

YEAR_MIN, YEAR_MAX = 2000, 2024
TIME_WINDOW = f"{YEAR_MIN}-{YEAR_MAX}"
YEAR_OUT = 2024

FAIL_LOUD = True  # raise on contract issues

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SCOPE_PATH:", SCOPE_PATH, "exists:", SCOPE_PATH.exists())
print("EMDAT_DIR:", EMDAT_DIR, "exists:", EMDAT_DIR.exists())
print("DESINV_DIR:", DESINV_DIR, "exists:", DESINV_DIR.exists())
print("POP_DIR:", POP_DIR, "exists:", POP_DIR.exists())
print("MASTER_PATH:", MASTER_PATH, "exists:", MASTER_PATH.exists())

PROJECT_ROOT: C:\pipelines\sewa-multihazar\notebooks
SCOPE_PATH: C:\pipelines\sewa-multihazar\data\raw\scope\ISO3_Regions.csv exists: True
EMDAT_DIR: C:\pipelines\sewa-multihazar\data\raw\EMDAT exists: True
DESINV_DIR: C:\pipelines\sewa-multihazar\data\raw\desinventar_sendai exists: True
POP_DIR: C:\pipelines\sewa-multihazar\data\raw\population exists: True
MASTER_PATH: C:\pipelines\sewa-multihazar\data\intermediate\wp3_country_indicators_long.csv exists: True


# 1) Contract for the master long table

In [3]:
HAZARDS_9 = [
    "Drought",
    "Flash Flooding",
    "Riverine Flooding Continued & Cluster",
    "Heatwave",
    "Storm Surge",
    "Wind",
    "Thunderstorm",
    "Wildfires",
    "Dust",
]
DIMENSIONS_5 = ["Prevalence", "Scale", "Impact", "Cascade impacts", "Future relevance"]

LONG_COLUMNS = [
    "iso3","country_name","region","hazard","dimension","source",
    "indicator_id","indicator_name","value_raw","unit_raw",
    "year","time_window","notes"
]
UPSERT_KEY = ["iso3","hazard","dimension","source","indicator_id","year"]

def hash_uid(parts) -> str:
    blob = "||".join([str(p) for p in parts])
    return md5(blob.encode("utf-8")).hexdigest()

def assert_contract(df: pd.DataFrame) -> None:
    missing = [c for c in LONG_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Contract violation: missing columns {missing}")
    bad_h = sorted(set(df["hazard"]) - set(HAZARDS_9))
    if bad_h:
        raise ValueError(f"Unknown hazards present: {bad_h}")
    bad_d = sorted(set(df["dimension"]) - set(DIMENSIONS_5))
    if bad_d:
        raise ValueError(f"Unknown dimensions present: {bad_d}")
    if df.duplicated(UPSERT_KEY).any():
        dup = df.loc[df.duplicated(UPSERT_KEY, keep=False), UPSERT_KEY].head(10)
        raise ValueError(f"Duplicate UPSERT keys found. Example rows:\n{dup}")

def upsert_to_master(new_df: pd.DataFrame, master_path: Path) -> pd.DataFrame:
    assert_contract(new_df)
    if master_path.exists():
        master = pd.read_csv(master_path)
        for c in LONG_COLUMNS:
            if c not in master.columns:
                master[c] = np.nan
        master = master[LONG_COLUMNS]
        key_master = master[UPSERT_KEY].astype(str).agg("||".join, axis=1)
        key_new = new_df[UPSERT_KEY].astype(str).agg("||".join, axis=1)
        master = master.loc[~key_master.isin(set(key_new))].copy()
        out = pd.concat([master, new_df[LONG_COLUMNS]], ignore_index=True)
    else:
        out = new_df[LONG_COLUMNS].copy()
    out.to_csv(master_path, index=False)
    return out

def find_first_col(cols, patterns) -> str | None:
    cols_l = {c: str(c).lower() for c in cols}
    for pat in patterns:
        rx = re.compile(pat, re.I)
        for c, cl in cols_l.items():
            if rx.search(cl):
                return c
    return None

# 2) Load scope list (ISO3 + region)

In [4]:
if not SCOPE_PATH.exists() and FALLBACKS["SCOPE"].exists():
    SCOPE_PATH = FALLBACKS["SCOPE"]

scope_raw = pd.read_csv(SCOPE_PATH)
scope_raw.columns = [c.replace("\xa0"," ").strip() for c in scope_raw.columns]

scope = scope_raw.copy()
scope["Region"] = scope["Region"].replace({np.nan: None}).ffill()
scope["Region"] = scope["Region"].astype(str).str.replace("\xa0","", regex=False).str.strip()
scope["Country"] = scope["Country"].astype(str).str.replace("\xa0","", regex=False).str.strip()
scope["ISO3.Code"] = scope["ISO3.Code"].astype(str).str.strip().str.upper()

scope = scope.rename(columns={"Region":"region","Country":"country_name","ISO3.Code":"iso3"})
scope = scope[["iso3","country_name","region"]].drop_duplicates()

if scope["iso3"].duplicated().any():
    raise ValueError("Duplicate ISO3 in scope list.")

print("Scope countries:", len(scope))
scope.head()

Scope countries: 47


,iso3,country_name,region
0,NGA,Nigeria,West Africa
1,GHA,Ghana,West Africa
2,SEN,Senegal,West Africa
3,BFA,Burkina Faso,West Africa
4,MLI,Mali,West Africa


# 3) Load population denominators (World Bank WDI SP.POP.TOTL zip)

In [5]:
def find_wdi_zip(pop_dir: Path) -> Path | None:
    if pop_dir.exists():
        zips = sorted(pop_dir.glob("API_SP.POP.TOTL*.zip"))
        if zips:
            return zips[0]
    return None

WDI_ZIP = find_wdi_zip(POP_DIR)
if WDI_ZIP is None and FALLBACKS["WDI_ZIP"].exists():
    WDI_ZIP = FALLBACKS["WDI_ZIP"]
if WDI_ZIP is None:
    raise FileNotFoundError("No WDI zip found for SP.POP.TOTL in data/raw/population or fallbacks.")

print("Using WDI zip:", WDI_ZIP)

def read_wdi_from_zip(zip_path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as z:
        candidates = [n for n in z.namelist() if n.lower().endswith(".csv") and "metadata" not in n.lower()]
        if not candidates:
            raise ValueError("No data CSV found in zip.")
        candidates_sorted = sorted(candidates, key=lambda n: (0 if Path(n).name.startswith("API_") else 1, len(n)))
        target = candidates_sorted[0]
        with z.open(target) as f:
            df = pd.read_csv(f, skiprows=4)
    return df

wdi = read_wdi_from_zip(WDI_ZIP)

year_cols = [c for c in wdi.columns if re.fullmatch(r"\d{4}", str(c))]
year_cols_int = sorted([int(c) for c in year_cols])
TARGET_YEAR = 2024
usable_years = [y for y in year_cols_int if y <= TARGET_YEAR]
if not usable_years:
    raise ValueError("No year columns <= 2024 in WDI file.")

wdi_pop = wdi.loc[wdi["Indicator Code"] == "SP.POP.TOTL"].copy()
for y in usable_years:
    wdi_pop[str(y)] = pd.to_numeric(wdi_pop[str(y)], errors="coerce")

def pick_latest_leq(row):
    for y in reversed(usable_years):
        v = row.get(str(y), np.nan)
        if pd.notna(v):
            return float(v), y
    return np.nan, None

vals = wdi_pop.apply(pick_latest_leq, axis=1, result_type="expand")
wdi_pop["pop_ref"] = vals[0]
wdi_pop["pop_ref_year"] = vals[1].astype("Int64")

pop_ref = (
    wdi_pop.rename(columns={"Country Code":"iso3","Country Name":"country_name_wdi"})[
        ["iso3","country_name_wdi","pop_ref","pop_ref_year"]
    ].copy()
)
pop_ref["iso3"] = pop_ref["iso3"].astype(str).str.strip().str.upper()
pop_ref = pop_ref.merge(scope[["iso3"]], on="iso3", how="inner")

pop_ref_path = OUT_DIR / "pop_wdi_sp_pop_totl_ref_2024.csv"
pop_ref.to_csv(pop_ref_path, index=False)

print("Pop rows (scope):", len(pop_ref), "missing pop_ref:", pop_ref["pop_ref"].isna().sum())
pop_ref.head()

Using WDI zip: C:\pipelines\sewa-multihazar\data\raw\population\API_SP.POP.TOTL_DS2_en_csv_v2_34.zip
Pop rows (scope): 46 missing pop_ref: 0


,iso3,country_name_wdi,pop_ref,pop_ref_year
0,AGO,Angola,37885849.0,2024
1,BDI,Burundi,14047786.0,2024
2,BEN,Benin,14462724.0,2024
3,BFA,Burkina Faso,23548781.0,2024
4,BWA,Botswana,2521139.0,2024


In [6]:
pop_map = pop_ref.set_index("iso3")["pop_ref"]

# 4) Load EM-DAT

In [7]:
def find_emdat_xlsx(emdat_dir: Path) -> Path | None:
    if emdat_dir.exists():
        xls = sorted(emdat_dir.glob("*.xlsx"))
        if xls:
            return xls[0]
    return None

EMDAT_PATH = find_emdat_xlsx(EMDAT_DIR)
if EMDAT_PATH is None and FALLBACKS["EMDAT"].exists():
    EMDAT_PATH = FALLBACKS["EMDAT"]
if EMDAT_PATH is None:
    raise FileNotFoundError("No EM-DAT xlsx found in data/raw/EMDAT or fallbacks.")

print("Using EM-DAT:", EMDAT_PATH)
emdat = pd.read_excel(EMDAT_PATH)
print("EM-DAT shape:", emdat.shape)
emdat.head()

Using EM-DAT: C:\pipelines\sewa-multihazar\data\raw\EMDAT\emdat_public.xlsx


EM-DAT shape: (27324, 46)


,DisNo.,Historic,Classification Key,Disaster Group,Disaster Subgroup,Disaster Type,Disaster Subtype,External IDs,Event Name,ISO,Country,Subregion,Region,Location,Origin,Associated Types,OFDA/BHA Response,Appeal,Declaration,AID Contribution ('000 US$),Magnitude,Magnitude Scale,Latitude,Longitude,River Basin,Start Year,Start Month,Start Day,End Year,End Month,End Day,Total Deaths,No. Injured,No. Affected,No. Homeless,Total Affected,Reconstruction Costs ('000 US$),"Reconstruction Costs, Adjusted ('000 US$)",Insured Damage ('000 US$),"Insured Damage, Adjusted ('000 US$)",Total Damage ('000 US$),"Total Damage, Adjusted ('000 US$)",CPI,Admin Units,Entry Date,Last Update
0,1900-0003-USA,Yes,nat-met-sto-tro,Natural,Meteorological,Storm,Tropical cyclone,NaN,NaN,USA,United States of America,Northern America,Americas,Galveston (Texas),NaN,"Avalanche (Snow, Debris)",No,No,No,NaN,220.0,Kph,NaN,NaN,NaN,1900,9.0,8.0,1900,9.0,8.0,6000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30000.0,1131126.0,2.652223,NaN,2004-10-18,2023-10-17
1,1900-0005-USA,Yes,tec-ind-fir-fir,Technological,Industrial accident,Fire (Industrial),Fire (Industrial),NaN,NaN,USA,United States of America,Northern America,Americas,"Hoboken, New York, Piers",NaN,Explosion,No,No,No,NaN,NaN,m3,NaN,NaN,NaN,1900,6.0,30.0,1900,6.0,30.0,300.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.652223,NaN,2003-07-01,2023-09-25
2,1900-0006-JAM,Yes,nat-hyd-flo-flo,Natural,Hydrological,Flood,Flood (General),NaN,NaN,JAM,Jamaica,Latin America and the Caribbean,Americas,Saint James,NaN,NaN,No,No,No,NaN,NaN,Km2,NaN,NaN,NaN,1900,1.0,6.0,1900,1.0,6.0,300.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.652223,NaN,2003-07-01,2023-09-25
3,1900-0007-JAM,Yes,nat-bio-epi-vir,Natural,Biological,Epidemic,Viral disease,NaN,Gastroenteritis,JAM,Jamaica,Latin America and the Caribbean,Americas,Porus,NaN,NaN,No,No,No,NaN,NaN,Vaccinated,NaN,NaN,NaN,1900,1.0,13.0,1900,1.0,13.0,30.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.652223,NaN,2003-07-01,2023-09-25
4,1900-0008-JPN,Yes,nat-geo-vol-ash,Natural,Geophysical,Volcanic activity,Ash fall,NaN,NaN,JPN,Japan,Eastern Asia,Asia,NaN,NaN,NaN,No,No,No,NaN,NaN,NaN,NaN,NaN,NaN,1900,7.0,7.0,1900,7.0,7.0,30.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.652223,NaN,2003-07-01,2023-09-25


## 4a) EM-DAT column detection 

In [8]:
emdat_cols = list(emdat.columns)

col_iso = find_first_col(emdat_cols, [r"^iso$|\\biso\\b", r"iso3", r"country\\s*code"])
col_country = find_first_col(emdat_cols, [r"Country\\s*name|\\bcountry\\b", r"Country"])
col_disno = find_first_col(emdat_cols, [r"disno|dis\\s*no"])
col_year = find_first_col(emdat_cols, [r"\\byear\\b", r"Start\\s*Year", r"Start Year"])

col_event_name = find_first_col(emdat_cols, [r"event\\s*name", r"disaster\\s*name", r"name$"])

col_type = find_first_col(emdat_cols, [r"disaster\\s*type", r"Disaster Type"])
col_subtype = find_first_col(emdat_cols, [r"disaster\\s*subtype", r"Disaster Subtype"])
col_subsub = find_first_col(emdat_cols, [r"subsubtype|sub\\s*sub", r"Disaster Subsubtype"])

col_start_y = find_first_col(emdat_cols, [r"start\\s*year", r"Start Year"])
col_start_m = find_first_col(emdat_cols, [r"start\\s*month", r"Start Month"])
col_start_d = find_first_col(emdat_cols, [r"start\\s*day", r"Start Day"])
col_end_y = find_first_col(emdat_cols, [r"end\\s*year", r"End Year"])
col_end_m = find_first_col(emdat_cols, [r"end\\s*month", r"End Month"])
col_end_d = find_first_col(emdat_cols, [r"end\\s*day", r"End Day"])

col_deaths = find_first_col(emdat_cols, [r"total\\s*deaths", r"deaths"])
col_affected = find_first_col(emdat_cols, [r"total\\s*affected", r"Total Affected"])
col_injured = find_first_col(emdat_cols, [r"injured"])
col_homeless = find_first_col(emdat_cols, [r"homeless"])

col_dmg_adj = find_first_col(emdat_cols, [r"total\\s*damage,*adjust", r"damage.*adjust", r"Total Damage"])
col_dmg_org = find_first_col(emdat_cols, [r"total\\s*damage,*original", r"damage.*original", r"total\\s*damage\\s*\\('000)", r"total damage"])

col_magnitude = find_first_col(emdat_cols, [r"\\bmagnitude\\b$", r"magnitude"])
col_magnitude_scale = find_first_col(emdat_cols, [r"magnitude\\s*scale", r"magnitude scale"])

col_admin_units = find_first_col(emdat_cols, [r"admin\\s*units", r"admin units"])

print("Detected columns:")
for k,v in {
    "iso":col_iso,"country":col_country,"disno":col_disno,"year":col_year,
    "event_name":col_event_name,
    "type":col_type,"subtype":col_subtype,"subsub":col_subsub,
    "start_y":col_start_y,"start_m":col_start_m,"start_d":col_start_d,
    "end_y":col_end_y,"end_m":col_end_m,"end_d":col_end_d,
    "deaths":col_deaths,"affected":col_affected,"injured":col_injured,"homeless":col_homeless,
    "damage_adj":col_dmg_adj,"damage_org":col_dmg_org,
    "magnitude":col_magnitude,"magnitude_scale":col_magnitude_scale,
    "admin_units":col_admin_units
}.items():
    print(f"{k:>14}: {v}")

if col_iso is None:
    raise ValueError("Could not detect ISO column in EM-DAT file.")

Detected columns:
           iso: ISO
       country: Country
         disno: DisNo.
          year: Start Year
    event_name: Event Name
          type: Disaster Type
       subtype: Disaster Subtype
        subsub: None
       start_y: Start Year
       start_m: Start Month
       start_d: Start Day
         end_y: End Year
         end_m: End Month
         end_d: End Day
        deaths: Total Deaths
      affected: Total Affected
       injured: No. Injured
      homeless: No. Homeless
    damage_adj: Insured Damage, Adjusted ('000 US$)
    damage_org: Total Damage ('000 US$)
     magnitude: Magnitude
magnitude_scale: Magnitude Scale
   admin_units: Admin Units


## 4b) EM-DAT cleaning + hazard mapping
### Mapping uses Disaster Type/Subtype/Subsubtype and (if present) Event Name text as a refinement.


In [9]:
em = emdat.copy()

em["iso3"] = em[col_iso].astype(str).str.strip().str.upper()
em["country_name_raw"] = em[col_country].astype(str).str.strip() if col_country else None

if col_year is not None:
    em["year"] = pd.to_numeric(em[col_year], errors="coerce").astype("Int64")
elif col_start_y is not None:
    em["year"] = pd.to_numeric(em[col_start_y], errors="coerce").astype("Int64")
else:
    raise ValueError("Could not infer year for EM-DAT.")

def build_date(y, m, d):
    try:
        if pd.isna(y): return pd.NaT
        y = int(y)
        m = 1 if (m is None or pd.isna(m)) else int(m)
        d = 1 if (d is None or pd.isna(d)) else int(d)
        return pd.Timestamp(year=y, month=max(1,min(12,m)), day=max(1,min(28,d)))
    except Exception:
        return pd.NaT

if col_start_y:
    em["start_date"] = [
        build_date(y,
                   em.loc[i, col_start_m] if col_start_m else None,
                   em.loc[i, col_start_d] if col_start_d else None)
        for i, y in enumerate(em[col_start_y].values)
    ]
else:
    em["start_date"] = pd.NaT

if col_end_y:
    em["end_date"] = [
        build_date(y,
                   em.loc[i, col_end_m] if col_end_m else None,
                   em.loc[i, col_end_d] if col_end_d else None)
        for i, y in enumerate(em[col_end_y].values)
    ]
else:
    em["end_date"] = pd.NaT

em["duration_days"] = (pd.to_datetime(em["end_date"]) - pd.to_datetime(em["start_date"])).dt.days
em.loc[em["duration_days"] < 0, "duration_days"] = np.nan

def col_to_num(df, c):
    if c is None:
        return pd.Series([np.nan]*len(df))
    return pd.to_numeric(df[c], errors="coerce")

em["deaths"] = col_to_num(em, col_deaths)
em["affected"] = col_to_num(em, col_affected)
em["injured"] = col_to_num(em, col_injured)
em["homeless"] = col_to_num(em, col_homeless)

# EM-DAT economic damages are in '000 US$ in the public table
em["damage_adj_usd_k"] = col_to_num(em, col_dmg_adj)
em["damage_org_usd_k"] = col_to_num(em, col_dmg_org)

em["magnitude"] = col_to_num(em, col_magnitude)
em["magnitude_scale"] = em[col_magnitude_scale].astype(str).str.strip() if col_magnitude_scale else None

em["disaster_type"] = em[col_type].astype(str).str.strip() if col_type else ""
em["disaster_subtype"] = em[col_subtype].astype(str).str.strip() if col_subtype else ""
em["disaster_subsubtype"] = em[col_subsub].astype(str).str.strip() if col_subsub else ""
em["event_name"] = em[col_event_name].astype(str).str.strip() if col_event_name else ""

def parse_admin_units(x):
    if pd.isna(x): 
        return (np.nan, np.nan)
    try:
        obj = json.loads(x) if isinstance(x, str) else x
        if not isinstance(obj, list):
            return (np.nan, np.nan)
        adm1 = set(); adm2 = set()
        for it in obj:
            if isinstance(it, dict):
                if it.get("adm1_code") is not None:
                    adm1.add(str(it.get("adm1_code")))
                if it.get("adm2_code") is not None:
                    adm2.add(str(it.get("adm2_code")))
        return (len(adm1), len(adm2))
    except Exception:
        return (np.nan, np.nan)

if col_admin_units:
    adm_counts = em[col_admin_units].apply(parse_admin_units)
    em["admin_spread_adm1"] = adm_counts.apply(lambda t: t[0])
    em["admin_spread_adm2"] = adm_counts.apply(lambda t: t[1])
else:
    em["admin_spread_adm1"] = np.nan
    em["admin_spread_adm2"] = np.nan

# Filter to window and scope
em = em.loc[(em["year"] >= YEAR_MIN) & (em["year"] <= YEAR_MAX)].copy()
em = em.merge(scope, on="iso3", how="inner")

def map_emdat_to_wp3(row):
    # Build a single text blob to refine mapping (type/subtype/subsub/event_name)
    blob = " | ".join([
        str(row.get("disaster_type","")),
        str(row.get("disaster_subtype","")),
        str(row.get("disaster_subsubtype","")),
        str(row.get("event_name","")),
    ]).lower()

    t  = str(row.get("disaster_type","")).lower()
    st = str(row.get("disaster_subtype","")).lower()
    sst= str(row.get("disaster_subsubtype","")).lower()

    # Exclude all technological/industrial disaster types before keyword matching
    if "industrial" in t or "technological" in t:
        return (None, "excluded_technological", "low")

    # Flood split
    if "flood" in blob:
        if "flash" in blob:
            return ("Flash Flooding", "flood_flash_keyword", "high")
        if "riverine" in blob:
            return ("Riverine Flooding Continued & Cluster", "flood_riverine_keyword", "high")
        if "coastal" in blob or "surge" in blob:
            return ("Storm Surge", "flood_coastal_or_surge_keyword", "high")
        # fallback flood
        return ("Riverine Flooding Continued & Cluster", "flood_fallback_unspecified", "low")

    # Drought
    if "drought" in blob:
        return ("Drought", "drought_keyword", "high")

    # Heatwave
    if ("extreme temperature" in t and "heat" in blob) or ("heat wave" in blob) or ("heatwave" in blob):
        return ("Heatwave", "heatwave_keyword", "high")

    # Wildfires
    if ("wildfire" in blob) or ("forest fire" in blob) or ("fire" in st and "wild" in blob):
        return ("Wildfires", "fire_keyword", "high")

    # Wind / storms
    if "storm" in blob or "cyclone" in blob or "tornado" in blob:
        # tropical cyclone family
        if any(k in blob for k in ["tropical cyclone","tropical storm","cyclone","typhoon","hurricane"]):
            return ("Wind", "storm_tropical_cyclone_keyword", "high")
        # tornado explicitly under Wind
        if "tornado" in blob:
            return ("Wind", "storm_tornado_keyword", "high")
        # lightning / thunderstorms
        if any(k in blob for k in ["lightning","thunderstorm","thunder"]):
            return ("Thunderstorm", "storm_thunder_keyword", "high")
        # dust storms
        if any(k in blob for k in ["sand","dust","sand/dust"]):
            return ("Dust", "storm_dust_keyword", "high")
        return ("Thunderstorm", "storm_generic_fallback", "low")

    # Dust
    if any(k in blob for k in ["dust","sand/dust","sandstorm"]):
        return ("Dust", "dust_keyword", "medium")

    return (None, "unmapped", "low")

em["hazard_wp3"], em["mapping_rule"], em["mapping_confidence"] = zip(*em.apply(map_emdat_to_wp3, axis=1))

print("EM-DAT mapped hazard counts:")
print(em["hazard_wp3"].value_counts(dropna=False).head(20))


EM-DAT mapped hazard counts:
hazard_wp3
None                                     2265
Riverine Flooding Continued & Cluster     756
Drought                                   164
Flash Flooding                            125
Wind                                      119
Thunderstorm                               98
Wildfires                                  20
Storm Surge                                 5
Heatwave                                    3
Name: count, dtype: int64


In [10]:
# 1) Everything that mapped to None
unmapped = em.loc[em["hazard_wp3"].isna()].copy()

# counts by type/subtype/subsubtype (most useful)
none_combos = (
    unmapped
    .groupby(["disaster_type", "disaster_subtype", "disaster_subsubtype"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)

print("\nTop 50 (None) combinations by frequency:")
print(none_combos.head(50).to_string(index=False))

print("\nUnique disaster_type among None (with counts):")
print(unmapped["disaster_type"].value_counts(dropna=False).head(50))

print("\nUnique disaster_subtype among None (with counts):")
print(unmapped["disaster_subtype"].value_counts(dropna=False).head(50))




Top 50 (None) combinations by frequency:
                   disaster_type                 disaster_subtype disaster_subsubtype   n
                            Road                             Road                     768
                        Epidemic                Bacterial disease                     363
                           Water                            Water                     340
                        Epidemic                    Viral disease                     177
            Fire (Miscellaneous)             Fire (Miscellaneous)                      84
                             Air                              Air                      78
           Collapse (Industrial)            Collapse (Industrial)                      62
             Mass movement (wet)                  Landslide (wet)                      61
Miscellaneous accident (General) Miscellaneous accident (General)                      50
                            Rail                          

In [11]:
# 2) Suspects: still None but contain keywords suggesting they might belong to WP3 hazards
# (This catches mapping bugs or unexpected EM-DAT naming.)
suspect_keywords = r"(flood|drought|heat|heat wave|wildfire|forest fire|fire|cyclone|tornado|thunder|lightning|dust|sand|surge|coastal)"
suspects = unmapped.loc[
    unmapped[["disaster_type","disaster_subtype","event_name"]]
    .astype(str)
    .agg(" | ".join, axis=1)
    .str.lower()
    .str.contains(suspect_keywords, regex=True, na=False)
].copy()

print("\nSuspects (None but keyword-matching) rows:", len(suspects))

suspect_combos = (
    suspects
    .groupby(["disaster_type","disaster_subtype","disaster_subsubtype"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)

print("\nTop 50 suspect (None) combos:")
print(suspect_combos.head(50).to_string(index=False))




Suspects (None but keyword-matching) rows: 204

Top 50 suspect (None) combos:
                disaster_type              disaster_subtype disaster_subsubtype  n
         Fire (Miscellaneous)          Fire (Miscellaneous)                     84
        Collapse (Industrial)         Collapse (Industrial)                     62
       Explosion (Industrial)        Explosion (Industrial)                     40
Industrial accident (General) Industrial accident (General)                     13
            Fire (Industrial)             Fire (Industrial)                      5


C:\Users\duruenaramirez\AppData\Local\Temp\ipykernel_1260\473171951.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(suspect_keywords, regex=True, na=False)


In [12]:
em_clean = em.loc[em["hazard_wp3"].notna()].copy()

if col_disno:
    em_clean["event_uid"] = "EMDAT:" + em_clean[col_disno].astype(str).str.strip() + ":" + em_clean["iso3"]
else:
    em_clean["event_uid"] = em_clean.apply(
        lambda r: "EMDAT_HASH:" + hash_uid([r["iso3"], r["year"], r["disaster_type"], r["disaster_subtype"], r["start_date"]]),
        axis=1,
    )

emdat_out = OUT_DIR / "emdat_clean_mapped_2000_2024.csv"
keep_cols = [
    "iso3","country_name","region","year","start_date","end_date","duration_days",
    "hazard_wp3","mapping_rule","mapping_confidence",
    "disaster_type","disaster_subtype","disaster_subsubtype","event_name",
    "deaths","injured","homeless","affected",
    "damage_org_usd_k","damage_adj_usd_k",
    "magnitude","magnitude_scale",
    "admin_spread_adm1","admin_spread_adm2",
    "event_uid",
]
keep_cols = [c for c in keep_cols if c in em_clean.columns]
em_clean[keep_cols].to_csv(emdat_out, index=False)
print("Wrote:", emdat_out, "rows:", len(em_clean))
em_clean[keep_cols].head()

Wrote: C:\pipelines\sewa-multihazar\data\intermediate\emdat_clean_mapped_2000_2024.csv rows: 1290


,iso3,country_name,region,year,start_date,end_date,duration_days,hazard_wp3,mapping_rule,mapping_confidence,disaster_type,disaster_subtype,disaster_subsubtype,event_name,deaths,injured,homeless,affected,damage_org_usd_k,damage_adj_usd_k,magnitude,magnitude_scale,admin_spread_adm1,admin_spread_adm2,event_uid
0,DJI,Djibouti,East Africa,2001,2001-06-01,2001-01-01,NaN,Drought,drought_keyword,high,Drought,Drought,,nan,NaN,NaN,NaN,100000.0,NaN,NaN,NaN,Km2,5.0,0.0,EMDAT:1999-9388-DJI:DJI
3,DJI,Djibouti,East Africa,2004,2004-04-12,2004-04-16,4.0,Flash Flooding,flood_flash_keyword,high,Flood,Flash flood,,nan,51.0,NaN,1500.0,100000.0,NaN,NaN,370.0,Km2,0.0,1.0,EMDAT:2004-0158-DJI:DJI
4,DJI,Djibouti,East Africa,2005,2005-04-01,2006-04-01,365.0,Drought,drought_keyword,high,Drought,Drought,,nan,NaN,NaN,NaN,150000.0,NaN,NaN,NaN,Km2,0.0,4.0,EMDAT:2005-9213-DJI:DJI
8,DJI,Djibouti,East Africa,2007,2007-01-01,2007-01-01,0.0,Drought,drought_keyword,high,Drought,Drought,,nan,NaN,NaN,NaN,42750.0,NaN,NaN,NaN,Km2,5.0,0.0,EMDAT:2007-9676-DJI:DJI
9,DJI,Djibouti,East Africa,2008,2008-07-01,2009-01-01,184.0,Drought,drought_keyword,high,Drought,Drought,,nan,NaN,NaN,NaN,340000.0,NaN,NaN,NaN,Km2,0.0,5.0,EMDAT:2008-9303-DJI:DJI


# 5) Load DesInventar combined translated file

In [13]:
def find_desinv_csv(desinv_dir: Path) -> Path | None:
    if desinv_dir.exists():
        csvs = sorted(desinv_dir.glob("*.csv"))
        if csvs:
            for p in csvs:
                n = p.name.lower()
                if "combined" in n or "translated" in n or "grouped" in n:
                    return p
            return csvs[0]
    return None

DESINV_PATH = find_desinv_csv(DESINV_DIR)
if DESINV_PATH is None and FALLBACKS["DESINV"].exists():
    DESINV_PATH = FALLBACKS["DESINV"]
if DESINV_PATH is None:
    raise FileNotFoundError("No DesInventar CSV found in data/raw/desinventar_sendai or fallbacks.")

print("Using DesInventar:", DESINV_PATH)
di = pd.read_csv(DESINV_PATH, low_memory=False)
print("DesInventar shape:", di.shape)
di.head()

Using DesInventar: C:\pipelines\sewa-multihazar\data\raw\desinventar_sendai\desinventar_combined_translated_grouped.csv
DesInventar shape: (72206, 69)


,serial,level0,level1,level2,name0,name1,name2,event_type_original,location,date_year,date_month,date_day,deaths,injured,missing,affected,homes_destroyed,homes_damaged,other_impacts,sources,local_value,usd_value,date_reported,date_damaged,has_deaths,has_injured,has_missing,has_affected,has_homes_destroyed,has_homes_damaged,has_other_impacts,relief,health,education,agriculture_livestock,industries,water_supply,sanitation,energy,communications,cause,cause_description,transportation,magnitude2,num_hospitals_affected,num_schools_affected,num_hectares_affected,num_livestock_affected,km_roads_affected,duration_days,displaced_persons,evacuated_persons,has_displaced_persons,has_evacuated_persons,has_relocated_persons,relocated_persons,clave,glide,defaultab,approved,latitude,longitude,uu_id,di_comments,country_code,event_type_original.1,event_type_translated,to_use,event_type_grouped
0,938,1.0,107.0,10701.0,Cabinda,Belize,Belize,INUNDAÇÃO,Cabinda,2006,1,31,0,0,0,0,27,18,NaN,Departamento de Redução de Riscos e Desastres,0.0,0.0,NaN,NaN,0,0,0,-1,-1,-1,0,0,0,0,0,0,0,0,0,0,Chuvas,Chuvas fortes,0,NaN,0,0,0.0,0,0.0,0,0,0,0,0,0,0,515,070/20,0.0,0,0.00000,0.000000,2a1612e3-0c5d-41ce-97e7-5198a48bbe41,No ano de 2006 O Município de Belize registou ...,ago,INUNDAÇÃO,FLOODING,yes,riverine flood
1,932,1.0,101.0,10101.0,Cabinda,Cabinda,Cabinda,INUNDAÇÃO,Cabinda,2002,1,31,0,0,0,0,0,0,NaN,Departamento de Redução de Riscos e Desastres,0.0,0.0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,Inundações,0,NaN,0,0,0.0,0,0.0,0,0,0,0,0,0,0,509,064/20,0.0,3,0.00000,0.000000,cdb497e9-2e22-4a64-9dda-591bc27b0c4c,No ano de 2002 o Município de Cabinda registou...,ago,INUNDAÇÃO,FLOODING,yes,riverine flood
2,922,1.0,103.0,10302.0,Cabinda,Cacongo(ex. Lândana),Dinge,INUNDAÇÃO,Cabinda,2010,1,31,0,0,0,0,0,0,NaN,Departamento de Redução de Riscos e Desastre,0.0,0.0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Chuvas,Inundações,0,NaN,0,0,0.0,0,0.0,0,0,0,0,0,0,0,500,056/20,0.0,3,0.00000,0.000000,a047e2ae-7c14-45de-85b4-cfe41d889740,No ano de 2010 a Comuna do Dinge Registou um t...,ago,INUNDAÇÃO,FLOODING,yes,riverine flood
3,13,4.0,417.0,41701.0,Luanda,Viana,Viana,INUNDAÇÃO,VIANA,2016,9,1,7,0,1,104,32,72,uma queda de arvore,COMANDO PROVINCIAL SNPCB LUANDA,0.0,0.0,EmilianoU3,2017-02-01,-1,0,-1,-1,-1,-1,0,0,0,0,0,0,0,0,0,0,Chuvas Fortes C/Ventos,NaN,0,NaN,0,0,0.0,0,0.0,0,0,1,0,-1,0,0,17,CCO18,0.0,0,-8.91971,13.389749,01528b79-29b2-4d5e-aff5-f6ca6aa59c44,NaN,ago,INUNDAÇÃO,FLOODING,yes,riverine flood
4,928,1.0,107.0,10701.0,Cabinda,Belize,Belize,INUNDAÇÃO,Cabinda,2010,1,31,0,0,0,0,0,0,NaN,Departamento de Redução de Riscos e Desastres,0.0,0.0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Chuvas,Inundações,0,NaN,0,0,0.0,0,0.0,0,0,0,0,0,0,0,505,060/20,0.0,3,0.00000,0.000000,2558bb5b-d095-42b5-b39f-1c03ad9abd7c,No Município de Belize registou-se um total de...,ago,INUNDAÇÃO,FLOODING,yes,riverine flood


## 5a) DesInventar column detection (expects `event_type_grouped` already prepared)


In [14]:
di_cols = list(di.columns)

col_iso_di = find_first_col(di_cols, [r"^iso3$|\\biso3\\b", r"^iso$", r"country_code"])
col_country_di = find_first_col(di_cols, [r"country"])
col_event_type_grp = find_first_col(di_cols, [r"event_type_grouped"])

col_date = find_first_col(di_cols, [r"event\\s*date", r"\\bdate\\b", r"fecha"])
col_date_year  = find_first_col(di_cols, [r"^date[_\s-]?year$", r"^event[_\s-]?year$"])
col_date_month = find_first_col(di_cols, [r"^date[_\s-]?month$", r"^event[_\s-]?month$"])
col_date_day   = find_first_col(di_cols, [r"^date[_\s-]?day$", r"^event[_\s-]?day$"])
col_year_di = find_first_col(di_cols, [r"\\byear\\b", r"anio", r"año"])

col_adm1 = find_first_col(di_cols, [r"adm1", r"name0"])
col_adm2 = find_first_col(di_cols, [r"adm2", r"name1"])

c_dead = find_first_col(di_cols, [r"\\bdead\\b|muert", r"deaths"])
c_injured = find_first_col(di_cols, [r"injur|sick|herid"])
c_affected = find_first_col(di_cols, [r"affected\\b|afect", r"affected"])

c_destroyed_h = find_first_col(di_cols, [r"destroyed\\s*houses|houses\\s*destroyed|viviendas\\s*destr", r"homes_destroyed"])
c_affected_h = find_first_col(di_cols, [r"affected\\s*houses|houses\\s*affected|viviendas\\s*afect", r"homes_damaged"])

c_evacuated = find_first_col(di_cols, [r"evacu"])
c_relocated = find_first_col(di_cols, [r"^relocated_persons$"])

# DesInventar distinguishes local currency ($) vs USD (US$) in some exports
c_loss_usd = find_first_col(di_cols, [r"loss\\s*value\\s*us\\$|loss.*u\\$|loss.*usd|value.*u\\$|\\bus\\$\\b|\\busd\\b", r"usd_value"])
c_loss_local = find_first_col(di_cols, [r"loss\\s*value\\s*\\$|loss.*\\$\\b(?!.*u\\$)|value.*\\$\\b(?!.*u\\$)|local\\s*currency", r"local_value"])

c_magnitude2 = find_first_col(di_cols, [r"magnitude2|magnitud"])
c_duration = find_first_col(di_cols, [r"duration_days|duration\\b|duraci"])

col_event_name_di = find_first_col(di_cols, [r"event\\s*name", r"comments", r"observ", r"description", r"descripcion"])

# If there is no single date column, construct one (keeps downstream cells intact)
if col_date is None and col_date_year is not None:
    # Build safe numeric components
    y = pd.to_numeric(di[col_date_year], errors="coerce")

    if col_date_month is not None:
        m = pd.to_numeric(di[col_date_month], errors="coerce")
    else:
        m = pd.Series(1, index=di.index, dtype="float")  # default Jan

    if col_date_day is not None:
        d = pd.to_numeric(di[col_date_day], errors="coerce")
    else:
        d = pd.Series(1, index=di.index, dtype="float")  # default day 1

    # Clamp month/day to valid ranges, coerce invalids to NaT
    parts = pd.DataFrame({
        "year": y,
        "month": m.fillna(1).clip(1, 12).astype("Int64"),
        "day": d.fillna(1).clip(1, 31).astype("Int64"),
    })

    # Create a real column in the raw df so later cells can do pd.to_datetime(d[col_date]) unchanged
    di["event_date"] = pd.to_datetime(parts, errors="coerce")
    col_date = "event_date"

    # If you don't have an explicit "year" column elsewhere, point col_year_di at date_year
    if col_year_di is None:
        col_year_di = col_date_year

print("Detected DesInventar columns:")
for k,v in {
    "iso3":col_iso_di, "country":col_country_di, "event_type_grouped":col_event_type_grp,
    "date":col_date, "year":col_year_di,
    "adm1":col_adm1, "adm2":col_adm2,
    "dead":c_dead, "injured":c_injured, "affected":c_affected,
    "destroyed_houses":c_destroyed_h, "affected_houses":c_affected_h,
    "evacuated":c_evacuated, "relocated":c_relocated,
    "loss_usd":c_loss_usd, "loss_local":c_loss_local,
    "magnitude2":c_magnitude2, "duration":c_duration,
    "event_name/comments":col_event_name_di,
    "date_year": col_date_year, "date_month": col_date_month, "date_day": col_date_day,
}.items():
    print(f"{k:>20}: {v}")

if col_event_type_grp is None:
    raise ValueError("DesInventar file missing event_type_grouped.")
if col_iso_di is None:
    raise ValueError("Could not detect ISO3 column in DesInventar file.")


Detected DesInventar columns:
                iso3: country_code
             country: country_code
  event_type_grouped: event_type_grouped
                date: event_date
                year: date_year
                adm1: name0
                adm2: name1
                dead: deaths
             injured: injured
            affected: affected
    destroyed_houses: homes_destroyed
     affected_houses: homes_damaged
           evacuated: evacuated_persons
           relocated: relocated_persons
            loss_usd: usd_value
          loss_local: local_value
          magnitude2: magnitude2
            duration: duration_days
 event_name/comments: di_comments
           date_year: date_year
          date_month: date_month
            date_day: date_day


## 5b) DesInventar cleaning + hazard mapping (based on event_type_grouped)


In [15]:
d = di.copy()
d["iso3"] = d[col_iso_di].astype(str).str.strip().str.upper()
d["country_name_raw"] = d[col_country_di].astype(str).str.strip() if col_country_di else None

if col_year_di:
    d["year"] = pd.to_numeric(d[col_year_di], errors="coerce").astype("Int64")
else:
    d["year"] = pd.Series([pd.NA]*len(d), dtype="Int64")

if col_date:
    d["event_date"] = pd.to_datetime(d[col_date], errors="coerce", dayfirst=True)
    d.loc[d["year"].isna() & d["event_date"].notna(), "year"] = d.loc[d["year"].isna() & d["event_date"].notna(), "event_date"].dt.year
else:
    d["event_date"] = pd.NaT

if c_duration:
    d["duration_days"] = pd.to_numeric(d[c_duration], errors="coerce")
else:
    d["duration_days"] = np.nan

if c_magnitude2:
    d["magnitude2"] = pd.to_numeric(d[c_magnitude2], errors="coerce")
else:
    d["magnitude2"] = np.nan

def maybe_num(col):
    return pd.to_numeric(d[col], errors="coerce") if col else pd.Series([np.nan]*len(d))

d["dead"] = maybe_num(c_dead)
d["injured"] = maybe_num(c_injured)
d["affected"] = maybe_num(c_affected)
d["destroyed_houses"] = maybe_num(c_destroyed_h)
d["affected_houses"] = maybe_num(c_affected_h)
d["evacuated"] = maybe_num(c_evacuated)
d["relocated"] = maybe_num(c_relocated)

d["loss_usd"] = maybe_num(c_loss_usd) if c_loss_usd else np.nan
d["loss_local"] = maybe_num(c_loss_local) if c_loss_local else np.nan

d["adm1"] = d[col_adm1].astype(str).str.strip() if col_adm1 else ""
d["adm2"] = d[col_adm2].astype(str).str.strip() if col_adm2 else ""

d["event_name_text"] = d[col_event_name_di].astype(str).str.strip() if col_event_name_di else ""

grp = d[col_event_type_grp].astype(str).str.strip().str.lower()
haz_map = {
    "drought": "Drought",
    "flash flood": "Flash Flooding",
    "riverine flood": "Riverine Flooding Continued & Cluster",
    "heatwave": "Heatwave",
    "storm surge": "Storm Surge",
    "wind": "Wind",
    "tropical cyclone": "Wind",
    "thunderstorm": "Thunderstorm",
    "wildfires": "Wildfires",
}
d["hazard_wp3"] = grp.map(haz_map)
d["mapping_rule"] = np.where(d["hazard_wp3"].notna(), "event_type_grouped", "unmapped")
d["mapping_confidence"] = np.where(d["hazard_wp3"].notna(), "high", "low")

# Filter to window and scope
d = d.loc[(d["year"] >= YEAR_MIN) & (d["year"] <= YEAR_MAX)].copy()
d = d.merge(scope, on="iso3", how="inner")

print("DesInventar mapped hazard counts:")
print(d["hazard_wp3"].value_counts(dropna=False).head(20))


DesInventar mapped hazard counts:
hazard_wp3
Drought                                  40127
Riverine Flooding Continued & Cluster    20243
Flash Flooding                            2749
Wildfires                                 2631
Wind                                      2564
Thunderstorm                                87
Heatwave                                    23
Storm Surge                                  2
Name: count, dtype: int64


## 5c) DesInventar event UID proxy 
### We build a stable proxy from iso3 + hazard + year + date + admin2 + truncated event name text.


In [16]:
def make_proxy_uid(row):
    return "DI_HASH:" + hash_uid([
        row.get("iso3",""),
        row.get("hazard_wp3",""),
        row.get("year",""),
        str(row.get("event_date",""))[:10],
        row.get("adm2",""),
        (row.get("event_name_text","") or "")[:80].lower(),
    ])

d["event_uid_proxy"] = d.apply(make_proxy_uid, axis=1)

di_out = OUT_DIR / "desinventar_clean_mapped_2000_2024.csv"
keep = [
    "iso3","country_name","region","year","event_date","hazard_wp3","mapping_rule","mapping_confidence",
    "adm1","adm2","duration_days","magnitude2",
    "dead","injured","affected","destroyed_houses","affected_houses",
    "evacuated","relocated","loss_usd","loss_local",
    "event_uid_proxy","event_name_text",
]
keep = [c for c in keep if c in d.columns]
d.loc[d["hazard_wp3"].notna(), keep].to_csv(di_out, index=False)
print("Wrote:", di_out, "rows:", (d["hazard_wp3"].notna()).sum())
d.loc[d["hazard_wp3"].notna(), keep].head()


Wrote: C:\pipelines\sewa-multihazar\data\intermediate\desinventar_clean_mapped_2000_2024.csv rows: 68426


,iso3,country_name,region,year,event_date,hazard_wp3,mapping_rule,mapping_confidence,adm1,adm2,duration_days,magnitude2,dead,injured,affected,destroyed_houses,affected_houses,evacuated,relocated,loss_usd,loss_local,event_uid_proxy,event_name_text
0,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,2006,2006-01-31,Riverine Flooding Continued & Cluster,event_type_grouped,high,Cabinda,Belize,0,NaN,0,0,0,27,18,0,0,0.0,0.0,DI_HASH:93b67ce397f4f3df97b44b8228b696f9,No ano de 2006 O Município de Belize registou ...
1,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,2002,2002-01-31,Riverine Flooding Continued & Cluster,event_type_grouped,high,Cabinda,Cabinda,0,NaN,0,0,0,0,0,0,0,0.0,0.0,DI_HASH:d75fe33b130dada641947037717c84a8,No ano de 2002 o Município de Cabinda registou...
2,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,2010,2010-01-31,Riverine Flooding Continued & Cluster,event_type_grouped,high,Cabinda,Cacongo(ex. Lândana),0,NaN,0,0,0,0,0,0,0,0.0,0.0,DI_HASH:6bd6a64b635557226f7410ec4b3d223a,No ano de 2010 a Comuna do Dinge Registou um t...
3,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,2016,2016-09-01,Riverine Flooding Continued & Cluster,event_type_grouped,high,Luanda,Viana,0,NaN,7,0,104,32,72,1,0,0.0,0.0,DI_HASH:6b7a2e7f159417e12457815372343463,nan
4,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,2010,2010-01-31,Riverine Flooding Continued & Cluster,event_type_grouped,high,Cabinda,Belize,0,NaN,0,0,0,0,0,0,0,0.0,0.0,DI_HASH:4b520d036ad0b7e327c7f4f5cda665e9,No Município de Belize registou-se um total de...


# 6) Exploration: coverage, completeness, overlap diagnostics

In [17]:
def coverage_table(df, dataset_name, hazard_col="hazard_wp3", year_col="year", uid_col=None):
    tmp = df.loc[df[hazard_col].notna()].copy()
    tmp["dataset"] = dataset_name
    if uid_col is None:
        tmp["uid"] = range(len(tmp))
        uid_col = "uid"
    cov = (
        tmp.groupby(["iso3","country_name","region",hazard_col,"dataset"], as_index=False)
           .agg(
               rows=("iso3","size"),
               years_with_any=(year_col, lambda s: s.nunique()),
               year_min=(year_col,"min"),
               year_max=(year_col,"max"),
               unique_events=(uid_col, lambda s: s.nunique()),
           )
           .rename(columns={hazard_col:"hazard"})
    )
    cov["time_window"] = TIME_WINDOW
    return cov

cov_em = coverage_table(em_clean, "EM-DAT", uid_col="event_uid")
cov_di = coverage_table(d, "DesInventar", uid_col="event_uid_proxy")

coverage = pd.concat([cov_em, cov_di], ignore_index=True)
coverage_path = OUT_DIR / "wp3_emdat_desinventar_coverage.csv"
coverage.to_csv(coverage_path, index=False)
print("Wrote:", coverage_path, "rows:", len(coverage))
coverage.head(20)

Wrote: C:\pipelines\sewa-multihazar\data\intermediate\wp3_emdat_desinventar_coverage.csv rows: 250


,iso3,country_name,region,hazard,dataset,rows,years_with_any,year_min,year_max,unique_events,time_window
0,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,EM-DAT,6,6,2001,2024,6,2000-2024
1,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Flash Flooding,EM-DAT,12,7,2015,2021,12,2000-2024
2,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Riverine Flooding Continued & Cluster,EM-DAT,32,17,2000,2023,32,2000-2024
3,BDI,Burundi,East Africa,Drought,EM-DAT,5,5,2003,2011,5,2000-2024
4,BDI,Burundi,East Africa,Flash Flooding,EM-DAT,6,5,2004,2018,6,2000-2024
5,BDI,Burundi,East Africa,Riverine Flooding Continued & Cluster,EM-DAT,26,15,2000,2024,26,2000-2024
6,BDI,Burundi,East Africa,Thunderstorm,EM-DAT,7,6,2004,2024,7,2000-2024
7,BEN,Benin,West Africa,Flash Flooding,EM-DAT,1,1,2013,2013,1,2000-2024
8,BEN,Benin,West Africa,Riverine Flooding Continued & Cluster,EM-DAT,11,10,2007,2024,11,2000-2024
9,BEN,Benin,West Africa,Thunderstorm,EM-DAT,1,1,2009,2009,1,2000-2024


In [18]:
def aggregate_emdat(df):
    x = df.copy()
    x["damage_adj_usd"] = x["damage_adj_usd_k"] * 1000.0
    agg = (
        x.groupby(["iso3","country_name","region","hazard_wp3"], as_index=False)
         .agg(
             events_total=("event_uid","nunique"),
             years_with_any=("year","nunique"),
             duration_mean=("duration_days","mean"),
             admin_spread_adm2_mean=("admin_spread_adm2","mean"),
             magnitude_mean=("magnitude","mean"),
             deaths_sum=("deaths","sum"),
             affected_sum=("affected","sum"),
             damage_adj_usd_sum=("damage_adj_usd","sum"),
         )
         .rename(columns={"hazard_wp3":"hazard"})
    )
    agg["events_per_year"] = agg["events_total"] / (YEAR_MAX - YEAR_MIN + 1)
    agg["deaths_per100k"] = (agg["deaths_sum"] / agg["iso3"].map(pop_map)) * 100000
    agg["affected_per100k"] = (agg["affected_sum"] / agg["iso3"].map(pop_map)) * 100000
    agg["damage_adj_usd_per_capita"] = (agg["damage_adj_usd_sum"] / agg["iso3"].map(pop_map))
    agg["source"] = "EM-DAT"
    return agg

def aggregate_desinv(df):
    x = df.loc[df["hazard_wp3"].notna()].copy()

    # admin spread proxy: mean distinct ADM2 per (proxy) event
    spread_proxy = (
        x.groupby(["iso3","hazard_wp3","event_uid_proxy"], as_index=False)
         .agg(adm2_n=("adm2", lambda s: s[s.notna() & (s.astype(str).str.strip()!="")].nunique()))
    )
    adm2_mean = spread_proxy.groupby(["iso3","hazard_wp3"], as_index=False).agg(admin_spread_adm2_mean=("adm2_n","mean"))

    agg = (
        x.groupby(["iso3","country_name","region","hazard_wp3"], as_index=False)
         .agg(
             events_total=("event_uid_proxy","nunique"),
             years_with_any=("year","nunique"),
             duration_mean=("duration_days","mean"),
             magnitude_mean=("magnitude2","mean"),
             deaths_sum=("dead","sum"),
             affected_sum=("affected","sum"),
             destroyed_houses_sum=("destroyed_houses","sum"),
             affected_houses_sum=("affected_houses","sum"),
             evacuated_sum=("evacuated","sum"),
             relocated_sum=("relocated","sum"),
             loss_usd_sum=("loss_usd","sum"),
         )
         .rename(columns={"hazard_wp3":"hazard"})
    )

    agg = agg.merge(adm2_mean.rename(columns={"hazard_wp3":"hazard"}), on=["iso3","hazard"], how="left")
    agg["events_per_year"] = agg["events_total"] / (YEAR_MAX - YEAR_MIN + 1)
    agg["deaths_per100k"] = (agg["deaths_sum"] / agg["iso3"].map(pop_map)) * 100000
    agg["affected_per100k"] = (agg["affected_sum"] / agg["iso3"].map(pop_map)) * 100000
    agg["loss_usd_per_capita"] = (agg["loss_usd_sum"] / agg["iso3"].map(pop_map))
    agg["source"] = "DesInventar"
    return agg

agg_em = aggregate_emdat(em_clean)
agg_di = aggregate_desinv(d)

agg_em.to_csv(OUT_DIR / "wp3_emdat_country_hazard_indicators_2000_2024.csv", index=False)
agg_di.to_csv(OUT_DIR / "wp3_desinventar_country_hazard_indicators_2000_2024.csv", index=False)

print("Aggregates:", len(agg_em), len(agg_di))
agg_em.head(10)

Aggregates: 165 85


,iso3,country_name,region,hazard,events_total,years_with_any,duration_mean,admin_spread_adm2_mean,magnitude_mean,deaths_sum,affected_sum,damage_adj_usd_sum,events_per_year,deaths_per100k,affected_per100k,damage_adj_usd_per_capita,source
0,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,6,6,177.833333,0.000000,NaN,58.0,7722216.0,0.0,0.24,0.153091,20382.850599,0.0,EM-DAT
1,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Flash Flooding,12,7,31.916667,1.727273,127300.950000,280.0,80587.0,0.0,0.48,0.739062,212.710028,0.0,EM-DAT
2,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Riverine Flooding Continued & Cluster,32,17,17.750000,1.387097,97327.293636,612.0,1272963.0,0.0,1.28,1.615379,3359.995971,0.0,EM-DAT
3,BDI,Burundi,East Africa,Drought,5,5,134.200000,1.400000,NaN,120.0,2412500.0,0.0,0.20,0.854227,17173.524711,0.0,EM-DAT
4,BDI,Burundi,East Africa,Flash Flooding,6,5,4.833333,2.166667,10956.250000,106.0,42882.0,0.0,0.24,0.754567,305.258067,0.0,EM-DAT
5,BDI,Burundi,East Africa,Riverine Flooding Continued & Cluster,26,15,15.160000,1.458333,7789.513333,155.0,439368.0,0.0,1.04,1.103377,3127.667235,0.0,EM-DAT
6,BDI,Burundi,East Africa,Thunderstorm,7,6,1.571429,1.666667,NaN,23.0,24210.0,0.0,0.28,0.163727,172.340325,0.0,EM-DAT
7,BEN,Benin,West Africa,Flash Flooding,1,1,15.000000,2.000000,NaN,0.0,32735.0,0.0,0.04,0.000000,226.340487,0.0,EM-DAT
8,BEN,Benin,West Africa,Riverine Flooding Continued & Cluster,11,10,33.000000,6.400000,NaN,148.0,1330518.0,0.0,0.44,1.023320,9199.636251,0.0,EM-DAT
9,BEN,Benin,West Africa,Thunderstorm,1,1,0.000000,1.000000,NaN,0.0,800.0,0.0,0.04,0.000000,5.531461,0.0,EM-DAT


In [19]:
def build_completeness(df_agg: pd.DataFrame) -> pd.DataFrame:
    mapping = {
        "events_total": "EVENTS_TOTAL",
        "events_per_year": "EVENTS_PER_YEAR",
        "duration_mean": "DURATION_MEAN_DAYS",
        "admin_spread_adm2_mean": "ADMIN_SPREAD_MEAN_ADM2",
        "magnitude_mean": "MAGNITUDE_MEAN",
        "deaths_sum": "DEATHS_SUM",
        "deaths_per100k": "DEATHS_PER100K",
        "affected_sum": "AFFECTED_SUM",
        "affected_per100k": "AFFECTED_PER100K",
        "damage_adj_usd_sum": "DAMAGE_ADJ_USD_SUM",
        "damage_adj_usd_per_capita": "DAMAGE_ADJ_USD_PER_CAPITA",
        "loss_usd_sum": "LOSS_USD_SUM",
        "loss_usd_per_capita": "LOSS_USD_PER_CAPITA",
        "destroyed_houses_sum": "DESTROYED_HOUSES_SUM",
        "affected_houses_sum": "AFFECTED_HOUSES_SUM",
        "evacuated_sum": "EVACUATED_SUM",
        "relocated_sum": "RELOCATED_SUM",
    }
    rows = []
    for col, ind in mapping.items():
        if col not in df_agg.columns:
            continue
        tmp = df_agg[["iso3","country_name","region","hazard","source","events_total","years_with_any"]].copy()
        tmp["indicator_id"] = ind
        tmp["value"] = df_agg[col].values
        tmp["non_null"] = tmp["value"].notna().astype(int)
        tmp["coverage_years_ratio"] = tmp["years_with_any"] / (YEAR_MAX - YEAR_MIN + 1)
        tmp["events_log"] = np.log1p(tmp["events_total"].fillna(0))
        # completeness score: presence + years coverage + scaled event density
        tmp["completeness_score"] = (1.0*tmp["non_null"] + 0.5*tmp["coverage_years_ratio"] + 0.25*tmp["events_log"])
        rows.append(tmp[["iso3","country_name","region","hazard","source","indicator_id","value","years_with_any","events_total","coverage_years_ratio","completeness_score"]])
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

comp = pd.concat([build_completeness(agg_em), build_completeness(agg_di)], ignore_index=True)
comp_path = OUT_DIR / "wp3_emdat_desinventar_indicator_completeness.csv"
comp.to_csv(comp_path, index=False)
print("Wrote:", comp_path, "rows:", len(comp))
comp.head(30)

Wrote: C:\pipelines\sewa-multihazar\data\intermediate\wp3_emdat_desinventar_indicator_completeness.csv rows: 3090


,iso3,country_name,region,hazard,source,indicator_id,value,years_with_any,events_total,coverage_years_ratio,completeness_score
0,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,EM-DAT,EVENTS_TOTAL,6.0,6,6,0.24,1.606478
1,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Flash Flooding,EM-DAT,EVENTS_TOTAL,12.0,7,12,0.28,1.781237
2,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Riverine Flooding Continued & Cluster,EM-DAT,EVENTS_TOTAL,32.0,17,32,0.68,2.214127
3,BDI,Burundi,East Africa,Drought,EM-DAT,EVENTS_TOTAL,5.0,5,5,0.20,1.547940
4,BDI,Burundi,East Africa,Flash Flooding,EM-DAT,EVENTS_TOTAL,6.0,5,6,0.20,1.586478
5,BDI,Burundi,East Africa,Riverine Flooding Continued & Cluster,EM-DAT,EVENTS_TOTAL,26.0,15,26,0.60,2.123959
6,BDI,Burundi,East Africa,Thunderstorm,EM-DAT,EVENTS_TOTAL,7.0,6,7,0.24,1.639860
7,BEN,Benin,West Africa,Flash Flooding,EM-DAT,EVENTS_TOTAL,1.0,1,1,0.04,1.193287
8,BEN,Benin,West Africa,Riverine Flooding Continued & Cluster,EM-DAT,EVENTS_TOTAL,11.0,10,11,0.40,1.821227
9,BEN,Benin,West Africa,Thunderstorm,EM-DAT,EVENTS_TOTAL,1.0,1,1,0.04,1.193287


In [20]:
# Approximate overlap (country+hazard+year, dates within 7 days)
emk = em_clean[["iso3","hazard_wp3","year","start_date","event_uid"]].copy()
emk = emk.rename(columns={"hazard_wp3":"hazard"})
emk["start_date"] = pd.to_datetime(emk["start_date"], errors="coerce")

dik = d.loc[d["hazard_wp3"].notna(), ["iso3","hazard_wp3","year","event_date","event_uid_proxy"]].copy()
dik = dik.rename(columns={"hazard_wp3":"hazard"})
dik["event_date"] = pd.to_datetime(dik["event_date"], errors="coerce")

cand = emk.merge(dik, on=["iso3","hazard","year"], how="inner")
cand["date_diff_days"] = (cand["start_date"] - cand["event_date"]).dt.days.abs()
cand["overlap_flag"] = np.where(cand["date_diff_days"].notna() & (cand["date_diff_days"] <= 7), 1, 0)

overlap = cand.groupby(["iso3","hazard"], as_index=False).agg(
    candidate_pairs=("overlap_flag","size"),
    likely_overlap_pairs=("overlap_flag","sum"),
)
overlap["likely_overlap_ratio"] = overlap["likely_overlap_pairs"] / overlap["candidate_pairs"]
overlap_path = OUT_DIR / "wp3_emdat_desinventar_overlap_diagnostics.csv"
overlap.to_csv(overlap_path, index=False)
print("Wrote:", overlap_path, "rows:", len(overlap))
overlap.sort_values("likely_overlap_pairs", ascending=False).head(20)


Wrote: C:\pipelines\sewa-multihazar\data\intermediate\wp3_emdat_desinventar_overlap_diagnostics.csv rows: 42


,iso3,hazard,candidate_pairs,likely_overlap_pairs,likely_overlap_ratio
28,NER,Riverine Flooding Continued & Cluster,13557,881,0.064985
10,GHA,Riverine Flooding Continued & Cluster,2536,365,0.143927
3,BFA,Riverine Flooding Continued & Cluster,891,318,0.356902
18,MOZ,Drought,1147,269,0.234525
13,KEN,Riverine Flooding Continued & Cluster,1884,191,0.101380
7,ETH,Riverine Flooding Continued & Cluster,2562,187,0.072990
20,MOZ,Wind,301,155,0.514950
24,MWI,Riverine Flooding Continued & Cluster,547,122,0.223035
39,UGA,Riverine Flooding Continued & Cluster,1334,110,0.082459
17,MLI,Riverine Flooding Continued & Cluster,836,80,0.095694


# 7) Automatic selection per country × hazard × indicator
### Rule: choose max(completeness_score), tie-break by years_with_any, then events_total.
### Additional tie-break: if indicator is adjusted damage, prefer EM-DAT.

In [21]:
def select_best(comp_df: pd.DataFrame) -> pd.DataFrame:
    x = comp_df.copy()
    x["prefer_emdat"] = np.where(
        (x["indicator_id"].isin(["DAMAGE_ADJ_USD_SUM","DAMAGE_ADJ_USD_PER_CAPITA"])) & (x["source"]=="EM-DAT"),
        1, 0
    )
    x = x.sort_values(
        ["iso3","hazard","indicator_id","completeness_score","years_with_any","events_total","prefer_emdat"],
        ascending=[True,True,True,False,False,False,False]
    )
    best = x.groupby(["iso3","hazard","indicator_id"], as_index=False).head(1).copy()
    x["rank"] = x.groupby(["iso3","hazard","indicator_id"]).cumcount()+1
    runner = x.loc[x["rank"]==2, ["iso3","hazard","indicator_id","source","completeness_score"]].rename(
        columns={"source":"runnerup_source","completeness_score":"runnerup_score"}
    )
    best = best.merge(runner, on=["iso3","hazard","indicator_id"], how="left")
    best["selection_rule"] = "max(completeness_score) tie-break (years_with_any, events_total, prefer_emdat_for_damage)"
    return best

decisions = select_best(comp)
decisions_path = OUT_DIR / "wp3_emdat_desinventar_selection_decisions.csv"
decisions.to_csv(decisions_path, index=False)
print("Wrote:", decisions_path, "rows:", len(decisions))
decisions.head(30)

Wrote: C:\pipelines\sewa-multihazar\data\intermediate\wp3_emdat_desinventar_selection_decisions.csv rows: 2622


,iso3,country_name,region,hazard,source,indicator_id,value,years_with_any,events_total,coverage_years_ratio,completeness_score,prefer_emdat,runnerup_source,runnerup_score,selection_rule
0,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,DesInventar,ADMIN_SPREAD_MEAN_ADM2,1.000000e+00,7,17,0.28,1.862593,0,EM-DAT,1.606478,max(completeness_score) tie-break (years_with_...
1,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,DesInventar,AFFECTED_HOUSES_SUM,6.346328e+06,7,17,0.28,1.862593,0,NaN,NaN,max(completeness_score) tie-break (years_with_...
2,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,DesInventar,AFFECTED_PER100K,1.919492e+03,7,17,0.28,1.862593,0,EM-DAT,1.606478,max(completeness_score) tie-break (years_with_...
3,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,DesInventar,AFFECTED_SUM,7.272160e+05,7,17,0.28,1.862593,0,EM-DAT,1.606478,max(completeness_score) tie-break (years_with_...
4,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,EM-DAT,DAMAGE_ADJ_USD_PER_CAPITA,0.000000e+00,6,6,0.24,1.606478,1,NaN,NaN,max(completeness_score) tie-break (years_with_...
5,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,EM-DAT,DAMAGE_ADJ_USD_SUM,0.000000e+00,6,6,0.24,1.606478,1,NaN,NaN,max(completeness_score) tie-break (years_with_...
6,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,DesInventar,DEATHS_PER100K,9.766179e-02,7,17,0.28,1.862593,0,EM-DAT,1.606478,max(completeness_score) tie-break (years_with_...
7,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,DesInventar,DEATHS_SUM,3.700000e+01,7,17,0.28,1.862593,0,EM-DAT,1.606478,max(completeness_score) tie-break (years_with_...
8,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,DesInventar,DESTROYED_HOUSES_SUM,1.000000e+01,7,17,0.28,1.862593,0,NaN,NaN,max(completeness_score) tie-break (years_with_...
9,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,DesInventar,DURATION_MEAN_DAYS,1.535294e+01,7,17,0.28,1.862593,0,EM-DAT,1.606478,max(completeness_score) tie-break (years_with_...


# 8) Build selected indicators in WP3 master long format and upsert


In [22]:
# Indicator metadata: map to WP3 dimensions
INDICATORS = [
    ("Prevalence", "EVENTS_TOTAL", "Events total (2000–2024)", "count_events"),
    ("Prevalence", "EVENTS_PER_YEAR", "Events per year (total / 25y)", "events_per_year"),
    ("Scale", "DURATION_MEAN_DAYS", "Mean duration (days)", "days"),
    ("Scale", "ADMIN_SPREAD_MEAN_ADM2", "Mean admin spread (ADM2 count)", "count_adm2"),
    ("Scale", "MAGNITUDE_MEAN", "Mean magnitude (hazard units; not standardized)", "hazard_units"),
    ("Impact", "DEATHS_SUM", "Deaths (sum)", "count_people"),
    ("Impact", "DEATHS_PER100K", "Deaths per 100k (Pop2024)", "per100k"),
    ("Impact", "AFFECTED_SUM", "Affected (sum)", "count_people"),
    ("Impact", "AFFECTED_PER100K", "Affected per 100k (Pop2024)", "per100k"),
    ("Impact", "DAMAGE_ADJ_USD_SUM", "Total damage, adjusted (USD)", "usd"),
    ("Impact", "DAMAGE_ADJ_USD_PER_CAPITA", "Total damage, adjusted (USD per person)", "usd_per_person"),
    ("Impact", "LOSS_USD_SUM", "Economic loss (USD) (sum)", "usd"),
    ("Impact", "LOSS_USD_PER_CAPITA", "Economic loss (USD per person)", "usd_per_person"),
    ("Impact", "DESTROYED_HOUSES_SUM", "Destroyed houses (sum)", "count_houses"),
    ("Impact", "AFFECTED_HOUSES_SUM", "Affected houses (sum)", "count_houses"),
    ("Cascade impacts", "EVACUATED_SUM", "Evacuated (sum)", "count_people"),
    ("Cascade impacts", "RELOCATED_SUM", "Relocated (sum)", "count_people"),
]
IND_META = {i[1]: (i[0], i[2], i[3]) for i in INDICATORS}

em_lookup = agg_em.set_index(["iso3","hazard"])
di_lookup = agg_di.set_index(["iso3","hazard"])

def get_value(source, iso3, hazard, indicator_id):
    if source == "EM-DAT":
        if (iso3, hazard) not in em_lookup.index:
            return np.nan
        row = em_lookup.loc[(iso3, hazard)]
        return {
            "EVENTS_TOTAL": row.get("events_total"),
            "EVENTS_PER_YEAR": row.get("events_per_year"),
            "DURATION_MEAN_DAYS": row.get("duration_mean"),
            "ADMIN_SPREAD_MEAN_ADM2": row.get("admin_spread_adm2_mean"),
            "MAGNITUDE_MEAN": row.get("magnitude_mean"),
            "DEATHS_SUM": row.get("deaths_sum"),
            "DEATHS_PER100K": row.get("deaths_per100k"),
            "AFFECTED_SUM": row.get("affected_sum"),
            "AFFECTED_PER100K": row.get("affected_per100k"),
            "DAMAGE_ADJ_USD_SUM": row.get("damage_adj_usd_sum"),
            "DAMAGE_ADJ_USD_PER_CAPITA": row.get("damage_adj_usd_per_capita"),
        }.get(indicator_id, np.nan)

    if source == "DesInventar":
        if (iso3, hazard) not in di_lookup.index:
            return np.nan
        row = di_lookup.loc[(iso3, hazard)]
        return {
            "EVENTS_TOTAL": row.get("events_total"),
            "EVENTS_PER_YEAR": row.get("events_per_year"),
            "DURATION_MEAN_DAYS": row.get("duration_mean"),
            "ADMIN_SPREAD_MEAN_ADM2": row.get("admin_spread_adm2_mean"),
            "MAGNITUDE_MEAN": row.get("magnitude_mean"),
            "DEATHS_SUM": row.get("deaths_sum"),
            "DEATHS_PER100K": row.get("deaths_per100k"),
            "AFFECTED_SUM": row.get("affected_sum"),
            "AFFECTED_PER100K": row.get("affected_per100k"),
            "LOSS_USD_SUM": row.get("loss_usd_sum"),
            "LOSS_USD_PER_CAPITA": row.get("loss_usd_per_capita"),
            "DESTROYED_HOUSES_SUM": row.get("destroyed_houses_sum"),
            "AFFECTED_HOUSES_SUM": row.get("affected_houses_sum"),
            "EVACUATED_SUM": row.get("evacuated_sum"),
            "RELOCATED_SUM": row.get("relocated_sum"),
        }.get(indicator_id, np.nan)

    return np.nan

scope_idx = scope.set_index("iso3")

rows = []
for _, r in decisions.iterrows():
    iso3 = r["iso3"]
    hazard = r["hazard"]
    ind = r["indicator_id"]
    src = r["source"]

    if ind not in IND_META:
        continue

    dim, ind_name, unit = IND_META[ind]
    val = get_value(src, iso3, hazard, ind)

    if pd.isna(val):
        continue

    # Skip DesInventar losses if missing/zero due to non-USD extraction issues
    # (we already only extract identified USD field, but keep a guard)
    if src == "DesInventar" and ind in {"LOSS_USD_SUM","LOSS_USD_PER_CAPITA"}:
        if pd.isna(val):
            continue

    notes = (
        f"Auto-selected source={src} using completeness_score (see wp3_emdat_desinventar_selection_decisions.csv). "
        f"Time window={TIME_WINDOW}. "
        "Prevalence computed as events_total / 25y. "
        "Per-capita uses WDI SP.POP.TOTL Pop2024 (or latest<=2024). "
        "Potential overlap exists; selection avoids double counting by keeping one source per indicator."
    )

    rows.append({
        "iso3": iso3,
        "country_name": scope_idx.loc[iso3, "country_name"],
        "region": scope_idx.loc[iso3, "region"],
        "hazard": hazard,
        "dimension": dim,
        "source": f"{src} (selected)",
        "indicator_id": f"{src.upper()}.{ind}_{TIME_WINDOW.replace('-','_')}",
        "indicator_name": ind_name,
        "value_raw": float(val) if pd.notna(val) else np.nan,
        "unit_raw": unit,
        "year": YEAR_OUT,
        "time_window": TIME_WINDOW,
        "notes": notes,
    })

selected_long = pd.DataFrame(rows)
if len(selected_long) == 0:
    raise ValueError("No selected indicators produced. Check hazard mapping and columns.")

assert_contract(selected_long)

selected_path = OUT_DIR / "wp3_selected_country_hazard_indicators_2000_2024.csv"
selected_long.to_csv(selected_path, index=False)
print("Wrote:", selected_path, "rows:", len(selected_long))

# Purge all existing master rows for this notebook's sources before upserting.
# upsert_to_master only removes rows whose keys appear in the new data, so hazards
# that were previously mapped but are now absent would otherwise persist as stale data.
if MASTER_PATH.exists():
    _master_pre = pd.read_csv(MASTER_PATH)
    _sources_to_purge = set(selected_long["source"].unique())
    _master_pre = _master_pre[~_master_pre["source"].isin(_sources_to_purge)].copy()
    _master_pre.to_csv(MASTER_PATH, index=False)
    print(f"Purged {_sources_to_purge} from master before upsert")

master = upsert_to_master(selected_long, MASTER_PATH)
print("Updated master rows:", len(master))
selected_long.head(30)

Wrote: C:\pipelines\sewa-multihazar\data\intermediate\wp3_selected_country_hazard_indicators_2000_2024.csv rows: 2486
Purged {'EM-DAT (selected)', 'DesInventar (selected)'} from master before upsert
Updated master rows: 3829


,iso3,country_name,region,hazard,dimension,source,indicator_id,indicator_name,value_raw,unit_raw,year,time_window,notes
0,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,Scale,DesInventar (selected),DESINVENTAR.ADMIN_SPREAD_MEAN_ADM2_2000_2024,Mean admin spread (ADM2 count),1.000000e+00,count_adm2,2024,2000-2024,Auto-selected source=DesInventar using complet...
1,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,Impact,DesInventar (selected),DESINVENTAR.AFFECTED_HOUSES_SUM_2000_2024,Affected houses (sum),6.346328e+06,count_houses,2024,2000-2024,Auto-selected source=DesInventar using complet...
2,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,Impact,DesInventar (selected),DESINVENTAR.AFFECTED_PER100K_2000_2024,Affected per 100k (Pop2024),1.919492e+03,per100k,2024,2000-2024,Auto-selected source=DesInventar using complet...
3,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,Impact,DesInventar (selected),DESINVENTAR.AFFECTED_SUM_2000_2024,Affected (sum),7.272160e+05,count_people,2024,2000-2024,Auto-selected source=DesInventar using complet...
4,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,Impact,EM-DAT (selected),EM-DAT.DAMAGE_ADJ_USD_PER_CAPITA_2000_2024,"Total damage, adjusted (USD per person)",0.000000e+00,usd_per_person,2024,2000-2024,Auto-selected source=EM-DAT using completeness...
5,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,Impact,EM-DAT (selected),EM-DAT.DAMAGE_ADJ_USD_SUM_2000_2024,"Total damage, adjusted (USD)",0.000000e+00,usd,2024,2000-2024,Auto-selected source=EM-DAT using completeness...
6,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,Impact,DesInventar (selected),DESINVENTAR.DEATHS_PER100K_2000_2024,Deaths per 100k (Pop2024),9.766179e-02,per100k,2024,2000-2024,Auto-selected source=DesInventar using complet...
7,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,Impact,DesInventar (selected),DESINVENTAR.DEATHS_SUM_2000_2024,Deaths (sum),3.700000e+01,count_people,2024,2000-2024,Auto-selected source=DesInventar using complet...
8,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,Impact,DesInventar (selected),DESINVENTAR.DESTROYED_HOUSES_SUM_2000_2024,Destroyed houses (sum),1.000000e+01,count_houses,2024,2000-2024,Auto-selected source=DesInventar using complet...
9,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,Scale,DesInventar (selected),DESINVENTAR.DURATION_MEAN_DAYS_2000_2024,Mean duration (days),1.535294e+01,days,2024,2000-2024,Auto-selected source=DesInventar using complet...



# 9) Flags
### - DesInventar losses can be local currency; this notebook only appends USD losses when explicitly detected.
### - Magnitude is not standardized across hazards; stored as-is for later standardization.
### - Overlap diagnostic is approximate; we avoid double counting by selection per indicator, not event-level dedup.
